In [ ]:
import kagglehub

# Download IAM Handwritten Forms dataset
path = kagglehub.dataset_download("naderabdalghani/iam-handwritten-forms-dataset")

print("Dataset downloaded to:", path)

Only use this when you want to change input dataset size

In [ ]:
!rm -rf PreprocessedImages

Dependencies and Conifgurations Initialization

In [2]:
# Dependencies
import os
from glob import glob
from collections import defaultdict
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive, files as FILES
import itertools
from tqdm import tqdm
import time
import io

In [66]:
# Training Configurations
RANDOMIZED = True  # Set to True to randomize data split and other stuff each run
BATCH_SIZE = 64
MAX_NO_OF_PAIRS_PER_EPOCH = 16204
IMAGE_DIMENSION = 512 # square image dimension
NO_OF_PARALLEL_LOAD_PROCESSES = 4 # Number of CPU processes that can load image data in parallel (caution: increases cpu usage and memory usage)
SEED = 42
SPLIT = 0.1 # Validation Split Size

if not RANDOMIZED :
    random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    GENERATOR = torch.Generator().manual_seed(SEED)

# Mount Google Drive
drive.mount('/content/drive')

# Paths
RAW_DATASET_PATH = path  # Define this before running
PREPROCESSED_PATH = "/content/PreprocessedImages"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Preprocess Images into Tensors with applied image processing

In [4]:
# Transform or Preprocessing on Image
transform = transforms.Compose([
    transforms.Resize((IMAGE_DIMENSION, IMAGE_DIMENSION)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


Pipeline to Make Labeled Pairs of Preprocessed Image Tensors for Training-Validation Dataset

In [6]:
# Step 1: Preprocess all images and save as tensors for Training
os.makedirs(PREPROCESSED_PATH, exist_ok=True)

for root, dirs, files in os.walk(RAW_DATASET_PATH):
    writer_id = os.path.basename(root)
    for file in files:
        if file.endswith(".png"):
            img_path = os.path.join(root, file)
            save_dir = os.path.join(PREPROCESSED_PATH, writer_id)
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, file.replace(".png", ".pt"))

            if not os.path.exists(save_path):
                img = Image.open(img_path).convert("L")
                tensor = transform(img)
                torch.save(tensor, save_path)

In [ ]:
# Dataset class used to make image pairs labeled with trust using preprocessed tensors
class HandwritingPairDataset(Dataset):
    def __init__(self, preprocessed_root, max_pairs):
        self.writer_to_tensors = defaultdict(list)
        for writer_id in os.listdir(preprocessed_root):
            writer_path = os.path.join(preprocessed_root, writer_id)
            if os.path.isdir(writer_path):
                tensor_files = glob(os.path.join(writer_path, "*.pt"))
                if len(tensor_files) >= 2:
                    self.writer_to_tensors[writer_id] = tensor_files

        writers = list(self.writer_to_tensors.keys())
        self.pairs, self.labels = [], []

        max_pos = max_pairs / 4
        max_neg = max_pos * 3
        pos_count = 0
        neg_count = 0

        for writer in writers:
            if pos_count >= max_pos:
                break
            images = self.writer_to_tensors[writer]
            for img1, img2 in itertools.combinations(images, 2):
                if pos_count >= max_pos:
                    break
                self.pairs.append((img1, img2))
                self.labels.append(1)
                pos_count += 1

        while neg_count < max_neg:
            writer1, writer2 = random.sample(writers, 2)
            img1 = random.choice(self.writer_to_tensors[writer1])
            img2 = random.choice(self.writer_to_tensors[writer2])
            self.pairs.append((img1, img2))
            self.labels.append(0)
            neg_count += 1

        combined = list(zip(self.pairs, self.labels))
        random.shuffle(combined)
        self.pairs, self.labels = zip(*combined)
        self.pairs, self.labels = list(self.pairs), list(self.labels)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path = self.pairs[idx]
        img1 = torch.load(img1_path)
        img2 = torch.load(img2_path)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img1, img2, label
    
# To Load a new Normal Dataset and split
def load_new_dataset () :

  full_dataset = HandwritingPairDataset(PREPROCESSED_PATH, max_pairs=MAX_NO_OF_PAIRS_PER_EPOCH)

  val_ratio = SPLIT
  val_size = int(len(full_dataset) * val_ratio)
  train_size = len(full_dataset) - val_size

  if RANDOMIZED :
    GENERATOR = torch.Generator()

  train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=GENERATOR)

  train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NO_OF_PARALLEL_LOAD_PROCESSES, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NO_OF_PARALLEL_LOAD_PROCESSES, pin_memory=True)

  return train_loader, val_loader


Model Design and Definition

In [8]:
# Model Definition
class ResNetEncoder(nn.Module):
    def __init__(self, embedding_dim=256):
        super(ResNetEncoder, self).__init__()
        base = resnet18(weights = ResNet18_Weights.IMAGENET1K_V1)
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.feature_extractor = nn.Sequential(*list(base.children())[:-1])
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, embedding_dim),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.embedding(x)
        return x

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.encoder = ResNetEncoder()
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x1, x2):
        out1 = self.encoder(x1)
        out2 = self.encoder(x2)
        combined = torch.cat([out1, out2], dim=1)
        return self.classifier(combined)


Scratch Training

In [ ]:
# Training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SiameseNetwork().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_f1 = 0.0
train_loader, val_loader = load_new_dataset()

epochs = 6

for epoch in range(epochs):
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    for img1, img2, labels in train_loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(img1, img2)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (outputs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []
    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).unsqueeze(1)
            outputs = model(img1, img2)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (outputs > 0.5).float()
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    val_acc = accuracy_score(val_labels, val_preds)
    precision = precision_score(val_labels, val_preds, zero_division=1)
    recall = recall_score(val_labels, val_preds, zero_division=1)
    f1 = f1_score(val_labels, val_preds, zero_division=1)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/content/drive/MyDrive/Models/siamese_best_512_f1.pth")
        print(f"Saved new best model at epoch {epoch+1} with F1: {f1:.4f}")

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} || "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


Resume Training

In [ ]:
# RESUME Training with previously saved model weights on same Dataset (same 8102 pairs)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load the model and pretrained weights
model = SiameseNetwork().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/Models/siamese_best_512_f1.pth"))

# Optimizer, scheduler, loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.BCELoss()

train_loader, val_loader = load_new_dataset()

epochs = 1

for epoch in range(epochs):
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    for img1, img2, labels in train_loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(img1, img2)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (outputs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []
    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).unsqueeze(1)
            outputs = model(img1, img2)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (outputs > 0.5).float()
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    val_acc = accuracy_score(val_labels, val_preds)
    precision = precision_score(val_labels, val_preds, zero_division=1)
    recall = recall_score(val_labels, val_preds, zero_division=1)
    f1 = f1_score(val_labels, val_preds, zero_division=1)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/content/drive/MyDrive/Models/siamese_best_512_f1.pth")
        print(f"Saved new best model at epoch {epoch+1} with F1: {f1:.4f}")

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} || "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


Fine-tuning Resume Training with Randomized Pairs every Epoch - Make sure you do RANDOMIZED = True before you run this!

In [ ]:
# RESUME Training with previously saved model weights on same Dataset (on Randomized MAX_NO_OF_PAIRS_PER_EPOCH pairs each Epoch)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load the model and pretrained weights
model_path = "/content/Models/model1_0.958.pth"
model = SiameseNetwork().to(device)
model.load_state_dict(torch.load(model_path))

# Optimizer, scheduler, loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.BCELoss()

best_f1 = 0.80
epochs = 1

for epoch in range(epochs):
    train_loader, val_loader = load_new_dataset()
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    for img1, img2, labels in train_loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(img1, img2)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (outputs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []
    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device).unsqueeze(1)
            outputs = model(img1, img2)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (outputs > 0.5).float()
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    val_acc = accuracy_score(val_labels, val_preds)
    precision = precision_score(val_labels, val_preds, zero_division=1)
    recall = recall_score(val_labels, val_preds, zero_division=1)
    f1 = f1_score(val_labels, val_preds, zero_division=1)

    torch.save(model.state_dict(), f"/content/Models/model_16204_{f1:.3f}.pth")

    scheduler.step()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} || "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

Testing with our Samples

In [128]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

# Load model and weights
model = SiameseNetwork().to(device)
model.load_state_dict(torch.load("/content/Models/model1_0.958.pth", map_location=device))
model.eval()

img1 = Image.open('image2.png')
img2 = Image.open('image1.png')

# Threshold (0.6582)
def predict (model, img1, img2) :

  threshold = 0.6582

  # Images are of valid formats like png, jpeg, jpg, etc.
  t1 = transform(img1).unsqueeze(0).to(device)
  t2 = transform(img2).unsqueeze(0).to(device)

  output = model(t1, t2)
  score = output.item()
  p = torch.tensor(score)
  logit = torch.log(p/(1-p))
  return True if logit >= threshold else False

# Predict similarity
with torch.no_grad():
    print("Prediction:", "Same Writer" if predict(model,img1,img2) else "Different Writers")


Device : cuda
Prediction: Same Writer
